# PDF Input Quickstart for Qwen3.5-27B

この Notebook は、英語論文 PDF の **全ページ** を対象にして、**text として入れる方法** と **画像化して vision input として入れる方法** をまとめたものです。


## 1. 前提
- Docker で SGLang API サーバが起動している
- `OPENAI_BASE_URL` を必要に応じて設定する
- PDF サンプルは `papers/` に配置済み


In [ ]:
import os
from pathlib import Path
from openai import OpenAI

BASE_URL = os.environ.get('OPENAI_BASE_URL', 'http://127.0.0.1:30000/v1')
PAPERS = Path('../papers')
print('Using base_url =', BASE_URL)
print('Papers:', sorted(p.name for p in PAPERS.glob('*.pdf')))
client = OpenAI(api_key='EMPTY', base_url=BASE_URL)
client


## 2. PDF の全ページから text を抽出して入れる


In [ ]:
from pypdf import PdfReader

pdf_path = PAPERS / 'attention_is_all_you_need.pdf'
reader = PdfReader(str(pdf_path))
text = '\n'.join(page.extract_text() or '' for page in reader.pages)
print('page_count =', len(reader.pages))
print('text_chars =', len(text))
print(text[:3000])


In [ ]:
resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[
        {'role': 'system', 'content': 'You are reading a full English deep learning paper extracted from PDF text.'},
        {'role': 'user', 'content': 'Summarize the paper in Japanese with sections for problem, method, and key ideas.\n\n' + text[:60000]},
    ],
    max_tokens=512,
)
resp


## 3. PDF の全ページを画像化して入れる


In [ ]:
import fitz  # pymupdf

pdf_path = PAPERS / 'deep_residual_learning.pdf'
doc = fitz.open(pdf_path)
out_dir = PAPERS / 'rendered' / 'deep_residual_learning_all_pages'
out_dir.mkdir(parents=True, exist_ok=True)
image_paths = []
for i in range(len(doc)):
    page = doc.load_page(i)
    pix = page.get_pixmap(matrix=fitz.Matrix(1.5, 1.5))
    img_path = out_dir / f'deep_residual_learning_page_{i+1:02d}.png'
    pix.save(img_path)
    image_paths.append(img_path)
print('rendered_pages =', len(image_paths))
print(image_paths[:3], '...')


In [ ]:
import base64
import mimetypes

content = [{'type': 'text', 'text': 'These are rendered pages from an English deep learning paper PDF. Summarize the paper in Japanese and mention the overall topic and architecture.'}]
for img_path in image_paths:
    mime = mimetypes.guess_type(img_path.name)[0] or 'image/png'
    image_url = 'data:' + mime + ';base64,' + base64.b64encode(img_path.read_bytes()).decode('utf-8')
    content.append({'type': 'image_url', 'image_url': {'url': image_url}})

resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[{'role': 'user', 'content': content}],
    max_tokens=512,
)
resp


## 4. 補足
- 全ページ text 抽出は本文全体の要約向き
- 全ページ画像化は図表・レイアウト・ページ全体の理解向き
- 全ページ画像入力は重いので、必要に応じて一部ページだけに減らしてもよい
